# EDA and Modeling — AI Revenue Recovery

This notebook inspects the **actual** CSV columns, performs EDA, trains churn models, and documents revenue-at-risk **estimates**.

Run from the project root so `src` imports resolve, or add the parent folder to `sys.path` as in the next cell.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.config import RAW_DATA_PATH, RANDOM_SEED
from src.data_preprocessing import (
    DATA_DICTIONARY,
    load_raw_data,
    profile_dataset,
    prepare_xy,
)
from src.eda import with_churn_flag, churn_by_plan
from src.train_model import train_and_compare

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Project root:", ROOT)
print("CSV:", RAW_DATA_PATH)

## 1. Inspect the dataset

Do not assume column names. Profile whatever is in the file.

In [ ]:
df = load_raw_data()
profile = profile_dataset(df)
print("Rows:", profile.n_rows, "Columns:", profile.n_columns)
print("Duplicates:", profile.duplicate_rows)
print("\nDtypes:\n", pd.Series(profile.dtypes))
print("\nMissing:\n", pd.Series(profile.missing_values))
print("\nUnique counts:\n", pd.Series(profile.nunique))
print("\nNumeric:", profile.numeric_columns)
print("Categorical:", profile.categorical_columns)
print("Date columns:", profile.date_columns)
print("Target:", profile.target_column)
print("IQR outlier counts:", profile.outlier_counts_iqr)
df.head()

## 2. Data dictionary

In [ ]:
pd.DataFrame(
    [{"column": k, "meaning": v} for k, v in DATA_DICTIONARY.items()]
)

## 3. Exploratory data analysis

Beginner view: we look at how often customers churn, how revenue sits by plan, and which behaviour fields move with churn.

In [ ]:
work = with_churn_flag(df)
print(work["churn"].value_counts())
print("Churn rate:", work["churn_flag"].mean())
print(churn_by_plan(df))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
work["churn"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#2ca02c", "#d62728"], title="Churn counts")
sns.boxplot(data=work, x="churn", y="monthly_fee", ax=axes[0, 1])
sns.barplot(data=work, x="plan_type", y="churn_flag", ax=axes[0, 2], errorbar=None)
axes[0, 2].set_title("Plan-wise churn rate")
sns.barplot(data=work, x="payment_failures", y="churn_flag", ax=axes[1, 0], errorbar=None)
axes[1, 0].set_title("Payment failures vs churn")
sns.boxplot(data=work, x="churn", y="avg_weekly_usage_hours", ax=axes[1, 1])
sns.boxplot(data=work, x="churn", y="last_login_days_ago", ax=axes[1, 2])
plt.tight_layout()
plt.show()

In [ ]:
num_cols = [
    "monthly_fee",
    "avg_weekly_usage_hours",
    "support_tickets",
    "payment_failures",
    "tenure_months",
    "last_login_days_ago",
    "churn_flag",
]
plt.figure(figsize=(8, 6))
sns.heatmap(work[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlation heatmap")
plt.show()

In [ ]:
px.histogram(work, x="tenure_months", color="churn", barmode="overlay", title="Tenure vs churn").show()
px.box(work, x="churn", y="support_tickets", title="Support tickets vs churn").show()
px.pie(work.groupby("plan_type", as_index=False)["monthly_fee"].sum(), names="plan_type", values="monthly_fee", title="Revenue by plan").show()

## 4. Features for modeling

`prepare_xy` adds same-row ratios only (no test-set statistics). The sklearn Pipeline inside `train_and_compare` fits imputers/scalers on the **training** fold only.

In [ ]:
X, y = prepare_xy(df)
print(X.columns.tolist())
print("Target rate:", float(y.mean()))
X.describe()

## 5. Train and compare models

This cell writes `models/churn_model.pkl`, `models/preprocessing.pkl`, metrics, and scored customers. It can take a minute.

In [ ]:
np.random.seed(RANDOM_SEED)
summary = train_and_compare(df)
print("Best model:", summary["best_model"])
print("Selection rule:", summary["selection_rule"])
pd.DataFrame(summary["metrics"])

## 6. Confusion matrices (test set)

In [ ]:
for name, cm in summary["confusion_matrices"].items():
    print(name, cm)

## 7. Revenue-at-risk reminder

`Expected Revenue at Risk = monthly_fee × 6 × churn_probability`

The file has no remaining-months field. Six months is a **documented assumption**, not historical contract data.

Recovery probability is a **business score**, not a second supervised model on recovery labels (those labels are absent).

In [ ]:
scored = pd.read_csv(ROOT / "data" / "processed" / "scored_customers.csv")
print(scored[["user_id", "churn_probability", "risk_category", "expected_revenue_at_risk", "recovery_probability", "recommended_action"]].head())
print("\nTotal estimated EAR:", scored["expected_revenue_at_risk"].sum())
print(scored["risk_category"].value_counts())
print(scored["recommended_action"].value_counts())

## 8. Presentation takeaways

- Inspect real columns first; this CSV has 10 fields and a `Yes`/`No` churn label.
- Prefer **recall** and **ROC-AUC** when a missed churner is lost revenue.
- Keep estimates labeled as estimates in the dashboard and viva.
- Next: `streamlit run app.py`